In [7]:
import yaml
def load_yaml(path:str)->dict:#function works
    with open(path,"r") as yaml_file:
        yaml_data=yaml.safe_load(yaml_file)
    return yaml_data["pipeline"]

load_yaml(r"C:/Projects/NLPpipline/conf/pipeline_v1.yaml")

{'name': 'nlp_pipeline_v1',
 'version': '1.0.0',
 'description': 'NLP text processing and classification pipeline',
 'language': 'en',
 'enable_ml_classifier': True,
 'enable_embeddings': True,
 'enable_vector_index': True,
 'stages': ['input',
  'preprocessing',
  'feature_extraction',
  'ml_classification',
  'embeddings',
  'postprocessing',
  'output']}

In [11]:
import os
import json
import hashlib
from pathlib import Path
from typing import Dict, Any, List, Optional
from datetime import datetime
from abc import ABC, abstractmethod

# Core libraries
import charset_normalizer
import magic

# File-specific processors
import pdfplumber
import fitz  # PyMuPDF
from docx import Document
from bs4 import BeautifulSoup
import markdown as md
import pandas as pd
import csv
from PIL import Image
import pytesseract

# Database
import sqlalchemy
from pymongo import MongoClient

# Compression
import zipfile
import tarfile
import gzip
class BaseReader(ABC):
    """Base class for all file readers"""
    
    def __init__(self, output_dir: str = "./jsonl_output"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.mime_detector = magic.Magic(mime=True)
    
    def detect_encoding(self, file_path: str) -> str:
        """Detect file encoding with fallback"""
        with open(file_path, 'rb') as f:
            raw_data = f.read(100000)
        
        try:
            result = charset_normalizer.from_bytes(raw_data).best()
            if result:
                return result.encoding
        except:
            pass
        
        # Fallback strategy
        for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
            try:
                raw_data.decode(encoding)
                return encoding
            except:
                continue
        
        return 'utf-8'
    
    def calculate_hash(self, file_path: str) -> str:
        """Calculate SHA256 hash"""
        sha256_hash = hashlib.sha256()
        with open(file_path, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    
    def get_base_metadata(self, file_path: str) -> Dict[str, Any]:
        """Get common metadata for all files"""
        path = Path(file_path)
        stat = path.stat()
        
        return {
            'file_path': str(path.absolute()),
            'file_name': path.name,
            'file_size_bytes': stat.st_size,
            'file_extension': path.suffix.lower(),
            'mime_type': self.mime_detector.from_file(str(path)),
            'file_hash_sha256': self.calculate_hash(file_path),
            'created_timestamp': datetime.fromtimestamp(stat.st_ctime).isoformat(),
            'modified_timestamp': datetime.fromtimestamp(stat.st_mtime).isoformat(),
            'processed_timestamp': datetime.now().isoformat(),
            'reader_class': self.__class__.__name__
        }
    
    def write_to_jsonl(self, data: Dict[str, Any], output_file: str):
        """Write single record to JSONL"""
        output_path = self.output_dir / output_file
        
        with open(output_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(data, ensure_ascii=False) + '\n')
    
    @abstractmethod
    def read(self, file_path: str) -> Dict[str, Any]:
        """Abstract method - must be implemented by subclasses"""
        pass
    
    def process_and_save(self, file_path: str, output_file: str = "output.jsonl"):
        """Process file and save to JSONL"""
        try:
            data = self.read(file_path)
            self.write_to_jsonl(data, output_file)
            return True
        except Exception as e:
            error_data = {
                'file_path': file_path,
                'error': str(e),
                'timestamp': datetime.now().isoformat(),
                'reader_class': self.__class__.__name__
            }
            self.write_to_jsonl(error_data, "errors.jsonl")
            return False
class TxtReader(BaseReader):
    """Reader for .txt, .text files"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        encoding = self.detect_encoding(file_path)
        
        with open(file_path, 'r', encoding=encoding, errors='replace') as f:
            content = f.read()
        
        # Extract RAW features (no preprocessing)
        lines = content.split('\n')
        
        return {
            **metadata,
            'encoding_detected': encoding,
            'raw_content': content,
            'raw_lines': lines,
            'line_count': len(lines),
            'char_count': len(content),
            'byte_count': len(content.encode(encoding, errors='replace')),
            'empty_lines': sum(1 for line in lines if not line.strip()),
            'max_line_length': max(len(line) for line in lines) if lines else 0,
            'contains_null_bytes': '\x00' in content,
            'contains_bom': content.startswith('\ufeff'),
            'line_endings': self._detect_line_endings(file_path),
            'whitespace_stats': {
                'spaces': content.count(' '),
                'tabs': content.count('\t'),
                'newlines': content.count('\n')
            }
        }
    
    def _detect_line_endings(self, file_path: str) -> str:
        """Detect line ending type"""
        with open(file_path, 'rb') as f:
            sample = f.read(1000)
        
        if b'\r\n' in sample:
            return 'CRLF (Windows)'
        elif b'\n' in sample:
            return 'LF (Unix)'
        elif b'\r' in sample:
            return 'CR (Mac)'
        return 'Unknown'

reader = TxtReader(output_dir="./test_output")
result = reader.read(r"C:\Users\advai\OneDrive\Desktop\NLP\Pipline.txt")

In [15]:

from pprint import pprint



reader = TxtReader(output_dir="./test_output")
result = reader.read(r"C:\Users\advai\OneDrive\Desktop\NLP\NLP.txt")
pprint(result)

{'byte_count': 210189,
 'char_count': 210189,
 'contains_bom': False,
 'contains_null_bytes': False,
 'created_timestamp': '2025-12-14T19:58:03.493267',
 'empty_lines': 299,
 'encoding_detected': 'mac_iceland',
 'file_extension': '.txt',
 'file_hash_sha256': 'b4619f2c990246077bbb713138f55c6f2dcbb3990b324f2060837202a3dfa596',
 'file_name': 'NLP.txt',
 'file_path': 'C:\\Users\\advai\\OneDrive\\Desktop\\NLP\\NLP.txt',
 'file_size_bytes': 212158,
 'line_count': 1970,
 'line_endings': 'CRLF (Windows)',
 'max_line_length': 899,
 'mime_type': 'text/plain',
 'modified_timestamp': '2025-12-14T20:02:49.156734',
 'processed_timestamp': '2025-12-17T12:32:18.938823',
 'raw_content': '\n'
                'Media Bias and Propaganda Detection System Design\n'
                '\n'
                'Token, Sentence & Document-Level Feature Infrastructure\n'
                '\n'
                'Mathematical Foundation\n'
                '\n'
                'At the core, each document is modeled as a seq

In [12]:
class MarkdownReader(BaseReader):
    """Reader for .md, .markdown files"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        encoding = self.detect_encoding(file_path)
        
        with open(file_path, 'r', encoding=encoding, errors='replace') as f:
            raw_markdown = f.read()
        
        # Convert to HTML for structure analysis
        html_content = md.markdown(raw_markdown, extensions=['extra', 'codehilite', 'toc'])
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Extract structure
        headings = []
        for tag in soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6']):
            headings.append({
                'level': int(tag.name[1]),
                'text': tag.get_text(strip=True)
            })
        
        # Extract code blocks
        code_blocks = []
        for pre in soup.find_all('pre'):
            code = pre.find('code')
            if code:
                code_blocks.append({
                    'language': code.get('class', [''])[0].replace('language-', ''),
                    'content': code.get_text()
                })
        
        # Extract links
        links = [{'text': a.get_text(), 'url': a.get('href')} 
                 for a in soup.find_all('a')]
        
        return {
            **metadata,
            'encoding_detected': encoding,
            'raw_markdown': raw_markdown,
            'rendered_html': html_content,
            'plain_text': soup.get_text(separator='\n'),
            'structure': {
                'headings': headings,
                'heading_count': len(headings),
                'code_blocks': code_blocks,
                'code_block_count': len(code_blocks),
                'links': links,
                'link_count': len(links),
                'list_count': len(soup.find_all(['ul', 'ol'])),
                'table_count': len(soup.find_all('table'))
            },
            'markdown_features': {
                'has_frontmatter': raw_markdown.startswith('---'),
                'has_code_fence': '```' in raw_markdown,
                'has_inline_code': '`' in raw_markdown,
                'has_blockquote': raw_markdown.count('> ') > 0,
                'has_images': '![' in raw_markdown
            }
        }


In [16]:
from pprint import pprint
reader1 = MarkdownReader(output_dir="./test_output")
result1 = reader1.read(r"C:\Projects\NLPpipline\README.md")
pprint(result1)

{'created_timestamp': '2025-11-27T16:36:15.613445',
 'encoding_detected': 'utf_8',
 'file_extension': '.md',
 'file_hash_sha256': '1ecc52a12f22decfdc3d42d2968e95e24db3bf7a9a2741ab3e5513a5ab25d0b0',
 'file_name': 'README.md',
 'file_path': 'C:\\Projects\\NLPpipline\\README.md',
 'file_size_bytes': 5366,
 'markdown_features': {'has_blockquote': True,
                       'has_code_fence': True,
                       'has_frontmatter': False,
                       'has_images': False,
                       'has_inline_code': True},
 'mime_type': 'text/plain',
 'modified_timestamp': '2025-11-30T15:38:22.208905',
 'plain_text': 'NLPpipeline\n'
               '\n'
               '\n'
               'A deterministic, configuration-driven NLP processing framework '
               'designed for high-integrity media and document analysis.\n'
               '\n'
               'The system emphasizes reproducibility, auditability, '
               'modularity, and strict schema-validated outp

In [17]:
class PDFReader(BaseReader):
    """Reader for .pdf files with comprehensive extraction"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        
        # Primary extraction with pdfplumber
        text_content = []
        tables = []
        page_data = []
        images_info = []
        
        try:
            with pdfplumber.open(file_path) as pdf:
                pdf_metadata = pdf.metadata or {}
                
                for page_num, page in enumerate(pdf.pages, 1):
                    # Extract text
                    page_text = page.extract_text() or ""
                    
                    # Extract tables
                    page_tables = page.extract_tables() or []
                    
                    # Page dimensions
                    page_info = {
                        'page_number': page_num,
                        'width': page.width,
                        'height': page.height,
                        'rotation': page.rotation,
                        'text_length': len(page_text),
                        'has_text': bool(page_text.strip()),
                        'table_count': len(page_tables)
                    }
                    
                    text_content.append(page_text)
                    tables.extend([{'page': page_num, 'data': table} for table in page_tables])
                    page_data.append(page_info)
                    
                    # Check for images
                    if page.images:
                        images_info.extend([
                            {'page': page_num, 'image_index': i, **img}
                            for i, img in enumerate(page.images)
                        ])
        
        except Exception as e:
            pdf_metadata = {}
            text_content = [f"pdfplumber extraction failed: {e}"]
        
        # Fallback to PyMuPDF if text is empty
        if not ''.join(text_content).strip():
            try:
                doc = fitz.open(file_path)
                text_content = [page.get_text() for page in doc]
                doc.close()
            except:
                pass
        
        full_text = '\n\n'.join(text_content)
        
        # OCR detection (if text is suspiciously short for page count)
        needs_ocr = len(full_text.strip()) < len(page_data) * 50
        
        return {
            **metadata,
            'pdf_metadata': pdf_metadata,
            'page_count': len(page_data),
            'pages': page_data,
            'raw_text_by_page': text_content,
            'full_raw_text': full_text,
            'tables': tables,
            'table_count': len(tables),
            'images': images_info,
            'image_count': len(images_info),
            'text_statistics': {
                'total_chars': len(full_text),
                'total_words': len(full_text.split()),
                'total_lines': len(full_text.split('\n')),
                'avg_chars_per_page': len(full_text) / len(page_data) if page_data else 0
            },
            'quality_indicators': {
                'needs_ocr': needs_ocr,
                'is_scanned': needs_ocr,
                'has_tables': len(tables) > 0,
                'has_images': len(images_info) > 0,
                'is_empty': not full_text.strip()
            },
            'extraction_method': 'pdfplumber_primary_pymupdf_fallback'
        }


In [18]:
from pprint import pprint
reader2 = PDFReader(output_dir="./test_output")
result2 = reader2.read(r"c:\Users\advai\Downloads\Spring 2026 Orientation Schedule.pdf")
pprint(result2)

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


{'created_timestamp': '2025-12-12T13:30:51.631588',
 'extraction_method': 'pdfplumber_primary_pymupdf_fallback',
 'file_extension': '.pdf',
 'file_hash_sha256': '094f10e9186e9a7bc31e8f922d11df4ffde44ce184dcb2c261c4efacd44c454b',
 'file_name': 'Spring 2026 Orientation Schedule.pdf',
 'file_path': 'c:\\Users\\advai\\Downloads\\Spring 2026 Orientation '
              'Schedule.pdf',
 'file_size_bytes': 185340,
 'full_raw_text': 'ORIENTATION\n'
                  'SCHEDULE\n'
                  'Time Session\n'
                  '11:45-12:00 pm Check-in\n'
                  '12:00-12:15 pm Welcome Presentation + Orientation Purpose & '
                  'Road Map\n'
                  '12:15-12:25 pm Move into Breakout Sessions\n'
                  '12:25-1:55pm Break-Out Sessions\n'
                  'Group A: Military-Affiliated Students (Harned Hall 110)\n'
                  'Group B: Non-Military-Affiliated Students (Norman '
                  'Worthington Conf. Ctr.)\n'
                 

In [19]:
class DocsReader(BaseReader):
    """Reader for .docx, .doc files"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        
        # Only .docx is supported directly
        if not file_path.endswith('.docx'):
            return {
                **metadata,
                'error': 'Only .docx format supported. Convert .doc to .docx first.',
                'supported_format': False
            }
        
        doc = Document(file_path)
        
        # Extract paragraphs with styles
        paragraphs = []
        for para in doc.paragraphs:
            paragraphs.append({
                'text': para.text,
                'style': para.style.name,
                'alignment': str(para.alignment) if para.alignment else None,
                'is_heading': para.style.name.startswith('Heading'),
                'runs': len(para.runs)
            })
        
        # Extract tables
        tables = []
        for table_idx, table in enumerate(doc.tables):
            table_data = {
                'table_index': table_idx,
                'rows': len(table.rows),
                'cols': len(table.columns),
                'data': []
            }
            
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
                table_data['data'].append(row_data)
            
            tables.append(table_data)
        
        # Extract document properties
        core_props = doc.core_properties
        
        # Get full text
        full_text = '\n\n'.join([p['text'] for p in paragraphs if p['text']])
        
        # Extract headings structure
        headings = [p for p in paragraphs if p['is_heading']]
        
        return {
            **metadata,
            'document_properties': {
                'title': core_props.title,
                'author': core_props.author,
                'subject': core_props.subject,
                'keywords': core_props.keywords,
                'created': core_props.created.isoformat() if core_props.created else None,
                'modified': core_props.modified.isoformat() if core_props.modified else None,
                'revision': core_props.revision
            },
            'paragraphs': paragraphs,
            'paragraph_count': len(paragraphs),
            'full_raw_text': full_text,
            'headings': headings,
            'heading_count': len(headings),
            'tables': tables,
            'table_count': len(tables),
            'sections': len(doc.sections),
            'text_statistics': {
                'total_chars': len(full_text),
                'total_words': len(full_text.split()),
                'total_lines': len(full_text.split('\n'))
            },
            'style_distribution': self._get_style_distribution(paragraphs)
        }
    
    def _get_style_distribution(self, paragraphs: List[Dict]) -> Dict[str, int]:
        """Count paragraph styles"""
        styles = {}
        for para in paragraphs:
            style = para['style']
            styles[style] = styles.get(style, 0) + 1
        return styles


In [20]:
from pprint import pprint
reader3 = DocsReader(output_dir="./test_output")
result3 = reader3.read(r"C:\Users\advai\Downloads\Resume.docx")
pprint(result3)

{'created_timestamp': '2025-11-15T01:13:08.740608',
 'document_properties': {'author': 'Sarva Advaith Narayana',
                         'created': '2025-03-04T14:16:00+00:00',
                         'keywords': '',
                         'modified': '2025-08-25T02:17:00+00:00',
                         'revision': 21,
                         'subject': '',
                         'title': ''},
 'file_extension': '.docx',
 'file_hash_sha256': 'cde91196a4664f3011d3a75ce734cecd33abc51998dc25b84b950abe688efcc5',
 'file_name': 'Resume.docx',
 'file_path': 'C:\\Users\\advai\\Downloads\\Resume.docx',
 'file_size_bytes': 64756,
 'full_raw_text': 'Summary\n'
                  '\n'
                  'I am a passionate student specializing in AI and Machine '
                  'Learning at Woxsen University and an exchange student to '
                  'Saint Martins Univertsity Lacey Washington, USA. Balancing '
                  'academics and leadership, I lead transformative projects

In [21]:
class CSVReader(BaseReader):
    """Reader for .csv, .tsv files"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        encoding = self.detect_encoding(file_path)
        
        # Detect delimiter
        with open(file_path, 'r', encoding=encoding) as f:
            sample = f.read(1024)
            sniffer = csv.Sniffer()
            try:
                dialect = sniffer.sniff(sample)
                delimiter = dialect.delimiter
            except:
                delimiter = ',' if file_path.endswith('.csv') else '\t'
        
        # Read with pandas for robust parsing
        try:
            df = pd.read_csv(file_path, encoding=encoding, delimiter=delimiter)
            
            # Get column info
            columns_info = []
            for col in df.columns:
                col_info = {
                    'name': col,
                    'dtype': str(df[col].dtype),
                    'null_count': int(df[col].isnull().sum()),
                    'unique_count': int(df[col].nunique()),
                    'sample_values': df[col].head(5).tolist()
                }
                columns_info.append(col_info)
            
            # Convert to records (all rows)
            records = df.to_dict('records')
            
            return {
                **metadata,
                'encoding_detected': encoding,
                'delimiter': delimiter,
                'row_count': len(df),
                'column_count': len(df.columns),
                'columns': df.columns.tolist(),
                'columns_info': columns_info,
                'data_types': df.dtypes.astype(str).to_dict(),
                'raw_data': records,
                'sample_data': records[:10],
                'statistics': {
                    'total_cells': df.size,
                    'null_cells': int(df.isnull().sum().sum()),
                    'memory_usage_bytes': int(df.memory_usage(deep=True).sum())
                },
                'data_quality': {
                    'has_null_values': df.isnull().any().any(),
                    'has_duplicates': df.duplicated().any(),
                    'duplicate_count': int(df.duplicated().sum())
                }
            }
        
        except Exception as e:
            # Fallback to basic CSV reading
            with open(file_path, 'r', encoding=encoding) as f:
                reader = csv.reader(f, delimiter=delimiter)
                rows = list(reader)
            
            return {
                **metadata,
                'encoding_detected': encoding,
                'delimiter': delimiter,
                'row_count': len(rows),
                'raw_data': rows,
                'error': f'Pandas parsing failed: {e}'
            }



In [24]:
from pprint import pprint
reader4 = CSVReader(output_dir="./test_output")
result4 = reader4.read(r"C:\Users\advai\Downloads\state_region.csv")
pprint(result4)

{'column_count': 4,
 'columns': ['State', 'State Code', 'Region', 'Division'],
 'columns_info': [{'dtype': 'object',
                   'name': 'State',
                   'null_count': 0,
                   'sample_values': ['Alaska',
                                     'Alabama',
                                     'Arkansas',
                                     'Arizona',
                                     'California'],
                   'unique_count': 51},
                  {'dtype': 'object',
                   'name': 'State Code',
                   'null_count': 0,
                   'sample_values': ['AK', 'AL', 'AR', 'AZ', 'CA'],
                   'unique_count': 51},
                  {'dtype': 'object',
                   'name': 'Region',
                   'null_count': 0,
                   'sample_values': ['West', 'South', 'South', 'West', 'West'],
                   'unique_count': 4},
                  {'dtype': 'object',
                   'name': 'Division

In [25]:
class JSONReader(BaseReader):
    """Reader for .json files"""
    
    def read(self, file_path: str) -> Dict[str, Any]:
        metadata = self.get_base_metadata(file_path)
        encoding = self.detect_encoding(file_path)
        
        with open(file_path, 'r', encoding=encoding) as f:
            raw_content = f.read()
        
        try:
            json_data = json.loads(raw_content)
            
            return {
                **metadata,
                'encoding_detected': encoding,
                'raw_json_string': raw_content,
                'parsed_json': json_data,
                'json_structure': {
                    'root_type': type(json_data).__name__,
                    'is_array': isinstance(json_data, list),
                    'is_object': isinstance(json_data, dict),
                    'size': len(json_data) if isinstance(json_data, (list, dict)) else 1,
                    'keys': list(json_data.keys()) if isinstance(json_data, dict) else None,
                    'depth': self._calculate_json_depth(json_data)
                },
                'byte_size': len(raw_content.encode(encoding))
            }
        
        except json.JSONDecodeError as e:
            return {
                **metadata,
                'encoding_detected': encoding,
                'raw_json_string': raw_content,
                'parse_error': str(e),
                'error_line': e.lineno,
                'error_column': e.colno
            }
    
    def _calculate_json_depth(self, obj, current_depth=0):
        """Calculate maximum depth of JSON structure"""
        if isinstance(obj, dict):
            if not obj:
                return current_depth
            return max(self._calculate_json_depth(v, current_depth + 1) for v in obj.values())
        elif isinstance(obj, list):
            if not obj:
                return current_depth
            return max(self._calculate_json_depth(item, current_depth + 1) for item in obj)
        else:
            return current_depth



In [ ]:
from pprint import pprint
reader5 = JSONReader(output_dir="./test_output")
result5 = reader5.read(r"C:\Users\advai\Downloads\augmented_python_errors.json")
pprint(result5)

{'byte_size': 405461207,
 'created_timestamp': '2025-04-04T10:02:16.955913',
 'encoding_detected': 'ascii',
 'file_extension': '.json',
 'file_hash_sha256': '641d25344602008b713dfe7bff272138e66f5afb1dc69396f5e96909e8d7cbc6',
 'file_name': 'augmented_python_errors.json',
 'file_path': 'C:\\Users\\advai\\Downloads\\augmented_python_errors.json',
 'file_size_bytes': 405461207,
 'json_structure': {'depth': 2,
                    'is_array': True,
                    'is_object': False,
                    'keys': None,
                    'root_type': 'list',
                    'size': 1324915},
 'mime_type': 'text/plain',
 'modified_timestamp': '2025-04-04T10:03:36.541471',
 'parsed_json': 